# RAG sul fascicolo: vettoriale o ibrida?

Giorno 3, pomeriggio. 50 minuti, ognuno al proprio portatile.

Costruite una ricerca sul fascicolo in tre modi, misurate quale trova il chunk giusto, poi fate rispondere un
modello con le fonti. Tutti i dati sono sintetici.

| Parte | Minuti | Che cosa si fa |
|---|---|---|
| 1. L'archivio | 10 | Dividere i documenti in chunk, calcolare un vettore per chunk |
| 2. Tre ricerche | 20 | Parole chiave, vettoriale, ibrida: per ogni domanda, il chunk atteso è tra i primi? |
| 3. La risposta con le fonti | 15 | I chunk trovati vanno a un modello; si controlla ogni affermazione |
| 4. La tabella | 5 | I vostri numeri, da incollare nel foglio condiviso |

Le celle si eseguono in ordine, con Maiusc+Invio. Dove trovate **Da fare**, cambiate la cella e rieseguitela.

## 1. L'archivio

### 1.1 I chunk

Il fascicolo si taglia in pezzi piccoli, i chunk: sono loro i risultati di una ricerca, e ognuno tiene il
suo riferimento, perché senza riferimento non c'è citazione.

In [ ]:
from rag import Archivio, parole, chunk_del_corpus

chunk = chunk_del_corpus()
print(len(chunk), "chunk da", len({r.split(":")[0] for r, _ in chunk}), "documenti\n")
for riferimento, testo in chunk[:4]:
    print(riferimento, "|", testo[:110])

Un chunk è una riga di un documento. Il riferimento `DOC-05:8` vuol dire: riga 8 del file di DOC-05. È ciò che
permette di citare la fonte.

Nei verbali le domande e le risposte stanno su righe diverse. Un chunk fatto della sola risposta («Sì, ero in
cucina con lei») non dice a che cosa risponde: per questo ogni risposta viaggia con la sua domanda.

In [ ]:
for riferimento, testo in chunk:
    if riferimento in ("DOC-02:8", "DOC-05:6", "DOC-05:8", "DOC-52:6"):
        print(riferimento, "|", testo, "\n")

Guardate le righe 6 e 8 di DOC-05: due IBAN che differiscono per due cifre scambiate. E DOC-52: un altro estratto
conto, di un'altra persona, con altri IBAN. Nel corpus ci sono cinque documenti così, di un'altra frode: servono a
vedere se la ricerca li confonde con i nostri. Un chunk è una riga sola, e una riga non dice di quale caso parla.

### 1.2 Le parole

La ricerca per parole chiave lavora sulle parole di ogni chunk. Come si dividono conta:

In [ ]:
print(parole("Che cosa registra il sito alle 21:58:12?"))
print(parole('15 198.51.100.23 - - [02/Mar/2025:21:58:12 +0100] "POST /verifica HTTP/1.1" 302'))

L'orario resta intero, e le parole che non distinguono niente («che», «cosa», «il») spariscono.

### 1.3 L'archivio

Adesso l'archivio: le parole di ogni chunk per la ricerca per parole chiave, e un vettore per chunk per quella
vettoriale.

Più sotto, la cella `archivio = Archivio()` scarica il modello di embedding (1 GB) se non l'avete già
scaricato stamattina: serve la rete, una volta sola.

In [ ]:
archivio = Archivio()
print(archivio.vettori.shape, "= chunk x numeri per chunk")

## 2. Tre ricerche

Una domanda, tre modi di cercare: per parole chiave, per significato (vettoriale), e le due insieme
(ibrida). Prima ciascuna da sola, poi il confronto. La funzione qui sotto stampa i primi risultati di una o più
ricerche.

In [ ]:
def mostra(domanda, modi=("parole", "vettori", "ibrida"), k=5):
    """Stampa i primi k chunk trovati per la domanda, per ciascuna ricerca in `modi`.

    Ogni riga è un chunk: il riferimento (documento e riga) e l'inizio del testo.
    """
    print(domanda)
    for modo in modi:
        print(f"\n  {modo}")
        for riferimento, testo in archivio.cerca(domanda, modo, k):
            print(f"    {riferimento:<10} {testo[:95]}")

### 2.1 Ricerca per parole chiave (BM25)

Ogni chunk riceve un punteggio per le parole che ha in comune con la domanda. Una parola rara nel fascicolo, come
un orario o un IBAN, pesa molto; una parola che compare ovunque pesa poco. Un chunk lungo non vince solo perché è
lungo. Trova ciò che è scritto esattamente; non sa che «coniuge» e «moglie» sono la stessa persona.

In [ ]:
mostra("Che cosa risulta dal telefono alle 10:41?", modi=["parole"])

**Risposta.** `DOC-04:11` è al primo posto: «10:41» è nella domanda e nel chunk, ed è una parola rara, quindi
pesa più di tutte le altre.

### 2.2 Ricerca vettoriale

Il modello di embedding trasforma ogni chunk, e la domanda, in 768 numeri: un vettore. Testi che dicono la stessa
cosa hanno vettori vicini, anche con parole diverse, anche in un'altra lingua. La ricerca prende i chunk con il
vettore più vicino a quello della domanda (similarità del coseno).

In [ ]:
mostra("Il coniuge era in casa durante la chiamata?", modi=["vettori"])

**Risposta.** `DOC-02:8` è al primo posto, anche se non ha nessuna parola in comune con la domanda: il verbale
dice «moglie» e «telefonata». Il significato è lo stesso, e i vettori sono vicini.

### 2.3 Ricerca ibrida

Le due classifiche si fondono: ogni chunk prende 1 / (60 + posto) da ciascuna ricerca, e i due punti si sommano
(reciprocal rank fusion). Un chunk in alto in tutte e due vince; uno in alto in una sola può restare sopra. Adesso
le tre ricerche insieme, sulle stesse due domande.

In [ ]:
mostra("Che cosa risulta dal telefono alle 10:41?")

La risposta sta in `DOC-04:11`. Chi la trova?

- **parole**: sì, primo risultato (l'orario «10:41» è nel chunk)
- **vettori**: no, non è tra i primi cinque
- **ibrida**: sì, primo risultato, grazie alle parole

Adesso una domanda che non usa le parole del documento:

In [ ]:
mostra("Il coniuge era in casa durante la chiamata?")

Il verbale dice «moglie» e «telefonata», non «coniuge» e «chiamata»: la risposta è in `DOC-02:8`.

- **parole**: no, nessuna parola della domanda è nel chunk
- **vettori**: sì, primo risultato (confronta il significato)
- **ibrida**: no, i chunk in cima per parole lo spingono fuori dai primi cinque

### 2.4 La vostra domanda

**Da fare.** Scrivete una vostra domanda sul caso e guardate che cosa trovano le tre ricerche.

Se non vi viene in mente, provate: `mostra("Quanto è durata la telefonata con il falso operatore?")`.

In [ ]:
mostra("Quanto è durata la telefonata con il falso operatore?")  # la vostra domanda

**Risposta, per la domanda suggerita.** La risposta c'è, in `DOC-04:10`: chiamata in entrata dal numero del falso operatore, «durata 17 minuti». Nessuna delle tre ricerche la trova tra i primi cinque: la riga del registro chiamate non dice «telefonata» né «operatore». In cima esce invece la denuncia, dove la signora dice di non ricordare l'ora della telefonata. Un modello che riceve quei chunk rischia di rispondere che il fascicolo non lo dice, mentre lo dice: in un documento che la ricerca non ha portato su.

### 2.5 Il banco di prova

Due domande non sono una misura. Il banco di prova ne ha quindici, ciascuna con il chunk in cui sta la risposta:
sei con un identificativo esatto, sei parafrasi, due in cui due documenti si contraddicono, una a cui il fascicolo
non risponde.

In [ ]:
from prova_banco import MODI, tabella

def stampa(k):
    """Misura le tre ricerche sulle quindici domande del banco, con i primi k risultati.

    Stampa due tabelle: domanda per domanda (il chunk atteso è tra i primi k? sì o no) e i conteggi per tipo di
    domanda. Restituisce i conteggi, che servono alla parte 4.
    """
    esiti, conteggi = tabella(archivio, k)
    print(f"Primi {k} risultati\n n  tipo             parole  vettori  ibrida  domanda")
    for e in esiti:
        if e["chunk_attesi"]:
            print(f"{e['n']:>2}  {e['tipo']:<16} " + "  ".join(f"{'sì' if e[m] else 'no':<6}" for m in MODI) + f"  {e['domanda'][:60]}")
    print("\ntipo             domande  parole  vettori  ibrida")
    for tipo, c in conteggi.items():
        print(f"{tipo:<16} {c['domande']:>7}  {c['parole']:>6}  {c['vettori']:>7}  {c['ibrida']:>6}")
    return conteggi

conteggi = stampa(5)

Leggete la tabella per tipo di domanda, non il totale.

- Sugli identificativi chi vince? Guardate la domanda 2: perché la ricerca vettoriale non trova un orario?
- Sulle parafrasi chi vince? Guardate la domanda 10: la chat è in inglese, la domanda in italiano.
- Le contraddizioni chiedono due chunk insieme, uno per versione. Chi li riporta tutti e due?
- Le domande 8 e 12 non le trova nessuno. Aprite la 12 con `mostra(...)`: che cosa trova al suo posto?

**Da fare.** Rifate la tabella con i primi 3 risultati. Cambia qualcosa? Che cosa vi dice su dove sta il chunk
giusto, quando una ricerca lo trova?

In [ ]:
conteggi_3 = stampa(3)

**Risposte.**

La tabella, misurata su un Mac con processore Apple (su un altro portatile un sì può diventare un no). È uguale con i primi 5 e con i primi 3:

| Tipo | Domande | Parole | Vettori | Ibrida |
|---|---|---|---|---|
| identificativo | 6 | 6 | 2 | 6 |
| parafrasi | 6 | 1 | 4 | 2 |
| contraddizione | 2 | 1 | 1 | 2 |
| totale | 14 | 8 | 7 | 10 |

- **Identificativi.** Vince la ricerca per parole, 6 su 6; la vettoriale ne trova 2. Domanda 2: un orario come «10:41» non ha un significato da confrontare, e la vettoriale porta su chunk con altri orari, come l'SMS delle 10:12.
- **Parafrasi.** Vince la vettoriale, 4 su 6. Domanda 10: la chat dice «Move the money to the safe account by noon». Tra italiano e inglese le parole non coincidono quasi mai, il significato sì: il modello di embedding è multilingue.
- **Contraddizioni.** Servono due chunk insieme, uno per versione. La ricerca per parole completa la 13, la vettoriale la 14: solo l'ibrida le completa tutte e due.
- **Domande 8 e 12.** La 12 chiede se la banca ha recuperato la prima somma. La risposta è in `DOC-10:6` («Non è revocabile; la richiesta di richiamo ha avuto esito negativo») e in `DOC-03:16`: nessuno dei due dice «recuperare» né «somma». Al loro posto tornano il richiamo del secondo bonifico (`DOC-10:9`) e l'altra frode (`DOC-54:7`, dove la somma «è rientrata»): un chunk che sembra rispondere, del caso sbagliato. La 8 chiede se la signora aveva capito il collegamento tra email e chiamata; la risposta è `DOC-11:7` («non aver collegato l'email alla telefonata»), con parole diverse e una negazione.
- **I primi 3.** Non cambia niente: quando una ricerca trova il chunk giusto, lo mette già tra i primi tre. Allargare a cinque non salva una ricerca che ha sbagliato strada.

In sintesi: nessuna ricerca basta da sola. L'ibrida ha il totale migliore (10 su 14) ed è l'unica che regge su tutti i tipi, ma sulle parafrasi perde contro la vettoriale.

## 3. La risposta con le fonti

I primi cinque chunk della ricerca ibrida vanno a un modello, con un'istruzione: rispondere solo con quei chunk,
citare la fonte di ogni affermazione, dire quando il fascicolo non basta.

Il modello è quello scritto in `LAB_MODEL` nel file `.env` di questa cartella. Se manca, o se la rete non va, il
notebook usa risposte già calcolate: il resto dell'esercizio non cambia.

### 3.1 Che cosa vede il modello

In [ ]:
import csv
from risposte import QUI, leggi_env, precalcolata, prompt, rispondi

leggi_env()
with (QUI / "banco.csv").open(encoding="utf-8") as file:
    banco = {d["n"]: d for d in csv.DictReader(file)}

print(prompt(banco["1"]["domanda"], archivio.cerca(banco["1"]["domanda"], "ibrida", 5)))

### 3.2 La risposta, e le fonti da controllare

Questo è tutto ciò che il modello vede. Adesso la risposta:

In [ ]:
async def chiedi(numero, tutto=False):
    """Fa rispondere il modello alla domanda `numero` del banco, e la stampa.

    Con tutto=False il modello riceve i primi 5 chunk della ricerca ibrida (RAG); con tutto=True l'intero
    fascicolo, senza ricerca. Se il modello non risponde (chiave mancante, rete), stampa la risposta già calcolata
    e il motivo. In fondo stampa i chunk dati al modello e quelli attesi, per il confronto.
    """
    domanda = banco[numero]["domanda"]
    chunk_per_il_modello = archivio.chunk if tutto else archivio.cerca(domanda, "ibrida", 5)
    try:
        risposta, origine = await rispondi(domanda, chunk_per_il_modello), "Risposta del modello, adesso."
    except Exception as errore:  # in aula l'esercizio continua anche senza modello
        risposta, quando = precalcolata(numero, "tutto" if tutto else "rag")
        origine = f"Il modello non ha risposto: {str(errore)[:200]}\nQuesta è una risposta già pronta, {quando}."
    print(f"Domanda {numero}: {domanda}\n\n[{origine}]\n\n{risposta}\n")
    if not tutto:
        print("Chunk dati al modello:", ", ".join(r for r, _ in chunk_per_il_modello))
    print("Chunk attesi:", banco[numero]["chunk_attesi"] or "nessuno: il fascicolo non lo dice")

await chiedi("1")

**Da fare.** Per ogni affermazione della risposta, aprite il chunk citato (la cella qui sotto) e controllate che
lo dica davvero. È la regola del corso: un risultato è un indizio, si controlla sulla fonte.

In [ ]:
def apri(riferimento):
    """Stampa il testo intero di un chunk, dato il riferimento (per esempio "DOC-05:8"): così si controlla una citazione."""
    print(riferimento, "|", dict(archivio.chunk)[riferimento])

apri("DOC-05:8")

**Risposta.** `DOC-05:8` è il bonifico del 5 marzo, 1.500 euro, verso l'IBAN che termina con **465**: è il movimento giusto. La trappola è `DOC-05:6`, il bonifico del 3 marzo, con l'IBAN che termina con **456**: due cifre scambiate. Una risposta che cita la riga 6, o tratta i due IBAN come lo stesso conto, è sbagliata, e lo si vede solo aprendo il chunk e confrontando le cifre una per una. Nella risposta già calcolata il modello cita la riga giusta, ma aggiunge che «le fonti non coincidono completamente»: una confusione inutile, nata proprio dall'IBAN quasi uguale.

### 3.3 Una contraddizione

Una contraddizione: due documenti danno due importi diversi per lo stesso bonifico. Il modello li riporta tutti e due?
E gli importi che cita sono tutti del nostro caso? Aprite ogni fonte con `apri(...)` e guardate di chi parla il documento.

In [ ]:
await chiedi("13")

**Risposta.** Il nostro fascicolo ha due versioni del primo bonifico: **4.200 euro** (denuncia `DOC-01:9`, marito `DOC-02:16`, movimenti del conto `DOC-05:6`, banca `DOC-10:6`) e **2.400 euro** (l'impiegata della filiale, `DOC-03:14`, che riporta ciò che le ha detto la signora). Una buona risposta le riporta tutte e due, con la fonte di ciascuna; il peso sta con 4.200, confermato dai movimenti e dalla banca. Tra i chunk dati al modello ci sono però anche due righe di `DOC-52`, l'estratto conto dell'altra frode: 3.100 euro, 17/02/2025. Nella risposta già calcolata il modello apre proprio con quella riga e la presenta come «il primo bonifico». Aprite `apri("DOC-52:6")`: l'IBAN non è nessuno di quelli del nostro caso. Una fonte vera, del caso sbagliato.

### 3.4 Una domanda senza risposta

La domanda a cui il fascicolo non risponde. Che cosa fa il modello? Un orario, qui, sarebbe inventato: tra i chunk
non ce n'è nessuno. Guardate però quali documenti cita: se compare una data di febbraio, viene dall'altra frode. Una
fonte vera, del caso sbagliato.

In [ ]:
await chiedi("15")

**Risposta.** Il fascicolo non lo dice. La signora non ricorda l'ora (`DOC-01:13`) e la banca scrive che gli orari di esecuzione saranno trasmessi su provvedimento dell'Autorità Giudiziaria (`DOC-10:12`). Una risposta corretta dice che il dato manca. Si può accettare una finestra dedotta dalle fonti, purché sia presentata come deduzione: il bonifico è stato fatto durante la telefonata iniziata alle 10:19 (`DOC-04:10`, `DOC-01:9`). Un orario preciso è inventato. Nella risposta già calcolata il modello dice giustamente che l'ora non c'è, ma cita la data del 17/02/2025 da `DOC-52:6`: è il primo bonifico dell'altra frode. La fonte esiste, il caso è sbagliato.

### 3.5 Quando la ricerca non trova il chunk giusto

La domanda 12, una delle due che nessuna ricerca trova. Il modello riceve cinque chunk sbagliati: che cosa risponde?

In [ ]:
await chiedi("12")

**Risposta.** No: il primo bonifico è istantaneo, non è revocabile, e la richiesta di richiamo ha avuto esito negativo (`DOC-10:6`, confermato da `DOC-03:16`). Ma nessuno dei due chunk arriva al modello. Il modello riceve invece `DOC-10:9`, sul richiamo del *secondo* bonifico (esito non ancora noto), e `DOC-54:7`, dell'altra frode, dove la somma «è rientrata». Con questi chunk la risposta onesta è «il fascicolo, per quanto vedo, non lo dice»; quella pericolosa è «sì, la somma è rientrata», presa dal caso sbagliato. Nella risposta già calcolata il modello dice che non lo sa, ma nella spiegazione mescola i due casi. Il problema qui non è il modello: è la ricerca, che non gli ha dato il chunk giusto.

### 3.6 Tutto il fascicolo nel prompt

Il confronto: gli stessi quesiti con **tutto il fascicolo** nel prompt, senza ricerca. Con quindici documenti ci
sta. Risponde meglio? E con quarantamila documenti?

In [ ]:
await chiedi("12", tutto=True)

In [ ]:
await chiedi("13", tutto=True)

**Risposta.** Con tutto il fascicolo nel prompt il modello vede anche `DOC-10:6`. Nelle risposte già calcolate: sulla 12 risponde che la prima somma non è stata recuperata; sulla 13 separa i due casi (4.200 euro per Giulia Neri, 3.100 per l'altra vittima) invece di mescolarli. Però non riporta i 2.400 euro di `DOC-03:14`: anche con tutte le fonti davanti, la contraddizione va cercata da chi legge. Qui il prompt intero funziona perché quindici documenti, circa 16.000 caratteri, ci stanno. Con quarantamila documenti non ci stanno, e anche dove ci stessero il costo e il tempo di ogni domanda crescerebbero con il fascicolo: per questo serve la ricerca, e serve che trovi il chunk giusto.

## 4. La tabella

### 4.1 Le righe per il foglio condiviso

Incollate questa riga nel foglio condiviso: la vediamo insieme sulla prossima slide.

In [ ]:
righe = []
for nome, c in (("primi 5", conteggi), ("primi 3", conteggi_3)):
    for tipo, valori in c.items():
        righe.append(f"{nome}\t{tipo}\t{valori['domande']}\t{valori['parole']}\t{valori['vettori']}\t{valori['ibrida']}")
print("risultati\ttipo\tdomande\tparole\tvettori\tibrida")
print("\n".join(righe))

### 4.2 Discussione

Quattro domande per la discussione:

1. Per quale tipo di domanda usereste ciascuna ricerca?
2. Dove la ricerca ibrida ha fatto peggio della migliore delle altre due, e perché?
3. Quando il chunk giusto non è tra quelli trovati, che cosa ha fatto il modello?
4. Una citazione che esiste basta? Che cosa manca a un chunk per sapere di quale caso parla?

**Risposte.**

1. **Quale ricerca per quale domanda.** Parole chiave per ciò che si scrive esattamente: IBAN, orari, indirizzi IP, codici. Vettoriale quando si cerca un fatto con parole proprie, o in un'altra lingua. Ibrida quando non si sa in anticipo che domanda arriverà: non è la migliore su ogni tipo, ma non crolla su nessuno.
2. **Dove l'ibrida ha fatto peggio.** Sulle parafrasi: 2 su 6, contro 4 della vettoriale (domande 7 e 10). I chunk sbagliati che hanno qualche parola in comune con la domanda salgono nella classifica per parole, e la fusione li tiene sopra il chunk giusto che la vettoriale aveva trovato.
3. **Che cosa ha fatto il modello senza il chunk giusto.** Dipende dal modello, e va letto sullo schermo. Nelle risposte già calcolate: sulla 15 ha detto che l'orario non c'è, ma ha preso una data dall'altra frode; sulla 12 ha detto che non lo sa, mescolando i due casi nella spiegazione. Un modello non può citare un chunk che non ha ricevuto.
4. **Una citazione che esiste basta?** No. Va aperta, e va guardato di chi parla il documento. A un chunk manca il contesto: di chi è il documento, di quale caso. Un sistema vero lo aggiunge a ogni chunk, per esempio con il titolo del documento e il numero del fascicolo, oppure filtra per fascicolo prima di cercare.